# Soil Program Planner — Iowa corn and soybeans

For the Soil Conservation Office: each winter, before the next season's program sign-ups, decide which counties get priority for soil-program outreach and funding to cut drought losses.
**Data:** USDA NASS county yields, U.S. Drought Monitor, NOAA weather and NRCS soil, 99 Iowa counties, 2000–2025 (Data to Harvest hackathon master dataset).

In [1]:
# The headline text lives only in soil_planner/soil_tool.py (HEADLINE). Its quoted figures are verified by the check cells in section 3.
import sys, warnings
sys.path.insert(0, "..")                      # repo root, so soil_planner and src import from notebooks/
warnings.filterwarnings("ignore")
from IPython.display import display, Markdown
from soil_planner import soil_tool as st
display(Markdown(st.HEADLINE))

**What we found.** Counties that lost more than their own normal in past droughts tended to lose more in the next ones. We built a drought-penalty score from 2000–2015 and tested it on 2016–2025 droughts it had never seen. Of the counties it ranked in the top third, about 45% (corn) and 49% (soybeans) were among the hardest hit, compared with 33% by chance. The corn ranking held up in every test year; the soybean ranking was less consistent.

**What we did not find.** Soil water storage leans toward smaller losses, but its effect could not be separated from zero, so we don't recommend targeting programs by storage alone.

**How to use this.** The priority list shows where drought has hurt most and is likely to hurt again. Use it as a starting point for site visits and outreach, not as proof that a soil program will reduce losses.

### How to run

From the repository root, in a terminal:

```bash
source .venv/bin/activate            # Windows: .venv\Scripts\activate
jupyter lab notebooks/Soil_Program_Planner.ipynb
```

Then choose **Kernel → Restart Kernel and Run All Cells**. The notebook takes one to two minutes; the evidence tables at the bottom recompute from the data each time.

To run it without opening it: `jupyter nbconvert --to notebook --execute --inplace notebooks/Soil_Program_Planner.ipynb`

Dropdown menus need the optional `ipywidgets` package (`pip install ipywidgets`). Without it the notebook still runs and shows the default views set in the **Settings** cell below. Code cells are collapsed; click the arrow or the "..." at the left of a cell to show the code.

### Settings

Only used when dropdowns are unavailable. Change a value and re-run the notebook to change the default views.

In [2]:
DEFAULT_CROP = "Corn"                           # "Corn" or "Soybeans"
DEFAULT_DEFINITION = "D2+ (severe or worse)"    # or "D3+ (extreme or worse)"
DEFAULT_COUNTY = "Polk County"                  # any Iowa county name, e.g. "Sioux County"

In [3]:
# Setup: load the Iowa data and compute the deployment drought penalty (all years 2000-2025). fips is kept as text.
import pandas as pd

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

ia = st.compute_penalties(st.load_iowa_with_lon(), split=False)     # deployment: one period, all 26 years
COUNTY_NAMES = sorted(ia[st.COL_COUNTY].unique())
DEFAULT_METRIC = list(st.METRICS)[0]                                 # "Drought penalty (2000-2025)"
assert DEFAULT_CROP in st.CROPS and DEFAULT_DEFINITION in st.DROUGHT_DEFS and DEFAULT_COUNTY in COUNTY_NAMES, "check the Settings cell"
print(f"Loaded {len(ia):,} Iowa county-years with a reported yield "
      f"({ia[st.COL_FIPS].nunique()} counties, {ia[st.COL_YEAR].min()}–{ia[st.COL_YEAR].max()}, corn and soybeans).")
print(f"Drought penalty: log yield relative to the statewide median, minus the county's own non-D2+ years within +/- {st.WINDOW} years "
      f"(at least {st.MIN_NORMAL_YEARS}); a county's score needs at least {st.MIN_D2_YEARS_FOR_SCORE} drought years with a penalty.")
print("Dropdown menus:", "available" if HAS_WIDGETS else "NOT available (ipywidgets not installed) — showing the default views from the Settings cell")

Loaded 4,907 Iowa county-years with a reported yield (99 counties, 2000–2025, corn and soybeans).
Drought penalty: log yield relative to the statewide median, minus the county's own non-D2+ years within +/- 3 years (at least 2); a county's score needs at least 3 drought years with a penalty.
Dropdown menus: NOT available (ipywidgets not installed) — showing the default views from the Settings cell


---
## 1. Where drought hits hardest: historical drought penalty

One map, one crop, one drought definition, one metric. Hover over a county for its name and all metrics, including soil water storage.

- **Drought year** = a growing season in which the worst U.S. Drought Monitor category covering more than 1% of the county reached the chosen level. D2 is "severe drought"; D3 is "extreme drought".
- **Drought penalty (2000–2025)**, the default: how much more a county loses in drought years than in its own nearby normal years. Each drought year's yield is compared with the statewide median that year, then with the county's own non-drought years within three years of it. The score averages those penalties over the county's drought years. Negative = the county loses more than usual in drought. This is the basis of the priority list below, because it is the only score that beat chance on held-out years (section 3).
- **Drought frequency** = share of the county's reported years that were drought years.
- **Drought-year yield loss** = the county's average yield anomaly in its drought years, relative to its own 26-year trend.
- **Soil water storage** = plant-available water the top 100 cm of the county's soil can hold (NRCS SSURGO). It does not change over time.

A **blank county** has too few drought years under the chosen definition (fewer than three with a penalty), or none at all for the other metrics. That is expected, especially for D3+. This map shows what has already happened. It is not a forecast and not a risk score.

In [4]:
def show_map(crop_label, def_label, metric_label):
    stats = st.county_drought_stats(ia, st.CROPS[crop_label], st.DROUGHT_DEFS[def_label])
    st.make_map(stats, st.METRICS[metric_label], metric_label, crop_label, def_label).show()

if HAS_WIDGETS:
    widgets.interact(show_map,
                     crop_label=widgets.Dropdown(options=list(st.CROPS), value=DEFAULT_CROP, description="Crop"),
                     def_label=widgets.Dropdown(options=list(st.DROUGHT_DEFS), value=DEFAULT_DEFINITION, description="Drought"),
                     metric_label=widgets.Dropdown(options=list(st.METRICS), value=DEFAULT_METRIC, description="Metric"))
else:
    display(Markdown(f"*Static views for **{DEFAULT_CROP}, {DEFAULT_DEFINITION}** (no dropdowns: ipywidgets not installed). "
                     "Change the Settings cell to see another crop or definition.*"))
    for metric_label in st.METRICS:
        show_map(DEFAULT_CROP, DEFAULT_DEFINITION, metric_label)

*Static views for **Corn, D2+ (severe or worse)** (no dropdowns: ipywidgets not installed). Change the Settings cell to see another crop or definition.*

### Priority list

Counties in the **top third by drought penalty** for each crop, using all years 2000–2025 and the D2+ definition: the counties that have lost the most, relative to their own normal years, when drought came. Sorted from the largest penalty. The number of drought years behind each score and the county's soil water storage are shown alongside. A county without a score had fewer than three drought years with a usable penalty.

In [5]:
for crop_label, crop in st.CROPS.items():
    dep = st.deployment_penalty_scores(ia, crop)
    top = st.priority_list(dep)
    no_score = dep.loc[~dep["has_score"], st.COL_COUNTY].tolist()
    display(Markdown(f"**{crop_label}: top {len(top)} of {int(dep['has_score'].sum())} scored counties** "
                     f"(no score for {len(no_score)}: {', '.join(no_score) if no_score else 'none'})"))
    view = top[["rank", st.COL_FIPS, st.COL_COUNTY, "penalty_score", "drought_years_used", "confidence", "drought_years_total", st.COL_AWS]].rename(columns={
        st.COL_FIPS: "FIPS", st.COL_COUNTY: "county", "penalty_score": "drought penalty (%)",
        "drought_years_used": "drought years behind score", "drought_years_total": "drought years (all)", st.COL_AWS: "soil water storage (mm)"})
    display(view.round({"drought penalty (%)": 1, "soil water storage (mm)": 0}).astype({"rank": int}).set_index("rank"))
    n_lower = int((top["confidence"] == "lower").sum())
    display(Markdown(f"Scores built on few drought years are less reliable; verify these counties with local information. "
                     f"*({n_lower} of {len(top)} {crop_label.lower()} counties above are marked 'lower': fewer than {st.LOW_CONFIDENCE_YEARS} drought years behind the score.)*"))

**Corn: top 33 of 99 scored counties** (no score for 0: none)

,FIPS,county,drought penalty (%),drought years behind score,confidence,drought years (all),soil water storage (mm)
rank,,,,,,,
1,19097,Jackson County,-12.9,4,lower,4,175.0
2,19051,Davis County,-12.9,10,standard,10,163.0
3,19149,Plymouth County,-12.3,8,standard,9,206.0
4,19161,Sac County,-9.8,11,standard,11,190.0
5,19173,Taylor County,-9.4,5,lower,6,178.0
6,19159,Ringgold County,-8.7,6,standard,7,167.0
7,19093,Ida County,-8.6,7,standard,7,202.0
8,19023,Butler County,-8.6,6,standard,6,163.0
9,19027,Carroll County,-8.5,9,standard,9,190.0


Scores built on few drought years are less reliable; verify these counties with local information. *(6 of 33 corn counties above are marked 'lower': fewer than 6 drought years behind the score.)*

**Soybeans: top 33 of 99 scored counties** (no score for 0: none)

,FIPS,county,drought penalty (%),drought years behind score,confidence,drought years (all),soil water storage (mm)
rank,,,,,,,
1,19077,Guthrie County,-10.8,5,lower,5,180.0
2,19001,Adair County,-10.1,8,standard,10,181.0
3,19013,Black Hawk County,-9.8,6,standard,6,161.0
4,19023,Butler County,-9.7,6,standard,6,163.0
5,19175,Union County,-9.7,9,standard,10,179.0
6,19029,Cass County,-9.6,8,standard,10,189.0
7,19055,Delaware County,-9.5,6,standard,6,171.0
8,19135,Monroe County,-9.5,11,standard,11,167.0
9,19161,Sac County,-9.3,11,standard,11,190.0


Scores built on few drought years are less reliable; verify these counties with local information. *(8 of 33 soybeans counties above are marked 'lower': fewer than 6 drought years behind the score.)*

### Table behind the map

All 99 counties for the selected crop and definition, sorted by drought penalty (largest first). Use this to look up a county or to copy the numbers.

In [6]:
def show_table(crop_label, def_label):
    stats = st.county_drought_stats(ia, st.CROPS[crop_label], st.DROUGHT_DEFS[def_label])
    view = (stats[[st.COL_FIPS, st.COL_COUNTY, "drought_penalty_pct", "penalty_years_used", "years_reported", "drought_years",
                   "drought_share", "drought_loss_pct", "normal_year_anom", st.COL_AWS, "drought_year_list"]]
            .rename(columns={st.COL_FIPS: "FIPS", st.COL_COUNTY: "county", "drought_penalty_pct": "drought penalty 2000-2025 (%)",
                             "penalty_years_used": "drought years used for penalty", "years_reported": "years reported",
                             "drought_years": "drought years", "drought_share": "share of years in drought",
                             "drought_loss_pct": "mean yield anomaly in drought years (%)",
                             "normal_year_anom": "mean yield anomaly in other years (%)",
                             st.COL_AWS: "soil water storage (mm)", "drought_year_list": "drought years (list)"})
            .sort_values("drought penalty 2000-2025 (%)", na_position="last")
            .reset_index(drop=True))
    display(view.round({"drought penalty 2000-2025 (%)": 1, "share of years in drought": 2,
                        "mean yield anomaly in drought years (%)": 1, "mean yield anomaly in other years (%)": 1,
                        "soil water storage (mm)": 0}))

if HAS_WIDGETS:
    widgets.interact(show_table,
                     crop_label=widgets.Dropdown(options=list(st.CROPS), value=DEFAULT_CROP, description="Crop"),
                     def_label=widgets.Dropdown(options=list(st.DROUGHT_DEFS), value=DEFAULT_DEFINITION, description="Drought"))
else:
    show_table(DEFAULT_CROP, DEFAULT_DEFINITION)

,FIPS,county,drought penalty 2000-2025 (%),drought years used for penalty,years reported,drought years,share of years in drought,mean yield anomaly in drought years (%),mean yield anomaly in other years (%),soil water storage (mm),drought years (list)
0,19097,Jackson County,-12.9,4,26,4,0.15,-15.1,2.8,175.0,"2003, 2005, 2012, 2023"
1,19051,Davis County,-12.9,10,25,10,0.40,-10.8,7.3,163.0,"2000, 2003, 2005, 2006, 2011, 2012, 2013, 2017, 2022, 2023"
2,19149,Plymouth County,-12.3,8,24,9,0.38,-10.7,6.4,206.0,"2000, 2006, 2007, 2012, 2013, 2017, 2021, 2022, 2023"
3,19161,Sac County,-9.8,11,26,11,0.42,-9.7,7.1,190.0,"2000, 2003, 2006, 2012, 2013, 2014, 2017, 2020, 2021, 2022, 2023"
4,19173,Taylor County,-9.4,5,25,6,0.24,-5.9,1.9,178.0,"2000, 2002, 2003, 2006, 2012, 2018"
...,...,...,...,...,...,...,...,...,...,...,...
94,19189,Winnebago County,6.0,5,26,5,0.19,-2.8,0.7,189.0,"2000, 2003, 2012, 2021, 2023"
95,19147,Palo Alto County,6.6,9,26,9,0.35,-0.9,0.5,179.0,"2000, 2003, 2006, 2007, 2012, 2020, 2021, 2022, 2023"
96,19059,Dickinson County,6.8,8,26,8,0.31,-1.2,0.5,182.0,"2000, 2003, 2006, 2007, 2012, 2020, 2021, 2023"
97,19063,Emmet County,6.9,7,23,7,0.30,-1.9,0.8,186.0,"2000, 2003, 2006, 2007, 2012, 2020, 2021"


---
## 2. County report card

Pick a county, crop and drought definition. The card shows a summary, the county's yield anomaly in every year with drought years in red, and the underlying rows (one per year). For each drought year the rows also show the county's yield relative to the state that year, its **normal baseline** (its own non-drought years within three years, also relative to the state) and the resulting **drought penalty**.

In [7]:
def show_report_card(county, crop_label, def_label):
    card = st.county_report_card(ia, county, st.CROPS[crop_label], st.DROUGHT_DEFS[def_label])
    display(card["summary"])
    st.anomaly_bar(card["rows"], county, crop_label, def_label).show()
    rows = card["rows"].rename(columns={
        st.COL_YEAR: "year", st.COL_YIELD: "yield (bu/acre)", st.COL_REL_STATE: "yield vs state median (%)",
        "normal_baseline": "normal baseline vs state (%)", "drought_penalty": "drought penalty (%)",
        st.COL_ANOM: "yield anomaly vs own trend (%)", "severity_label": "worst drought category",
        "in_drought": "drought year", st.COL_WEEKS_D2: "weeks at D2+", st.COL_DSCI: "mean drought index (0-500)",
        st.COL_PRECIP_ANOM: "season rain vs normal (%)", st.COL_HEAT: "days at or above 35 °C"})
    display(rows.round(1))

if HAS_WIDGETS:
    widgets.interact(show_report_card,
                     county=widgets.Dropdown(options=COUNTY_NAMES, value=DEFAULT_COUNTY, description="County"),
                     crop_label=widgets.Dropdown(options=list(st.CROPS), value=DEFAULT_CROP, description="Crop"),
                     def_label=widgets.Dropdown(options=list(st.DROUGHT_DEFS), value=DEFAULT_DEFINITION, description="Drought"))
else:
    display(Markdown(f"*Static view for **{DEFAULT_COUNTY}, {DEFAULT_CROP}, {DEFAULT_DEFINITION}**. Change the Settings cell to pick another county.*"))
    show_report_card(DEFAULT_COUNTY, DEFAULT_CROP, DEFAULT_DEFINITION)

*Static view for **Polk County, Corn, D2+ (severe or worse)**. Change the Settings cell to pick another county.*

,value
County,Polk County
FIPS,19153
Crop,CORN
Drought definition,severity >= 3 (D2+)
Years with a reported yield,26
Drought years,10
Share of years in drought,38%
Drought years (list),"2000, 2003, 2006, 2012, 2013, 2020, 2021, 2022, 2023, 2024"
Drought penalty score (2000-2025),+1.5%
Drought years used for the score,8


,year,yield (bu/acre),yield vs state median (%),normal baseline vs state (%),drought penalty (%),yield anomaly vs own trend (%),worst drought category,drought year,weeks at D2+,mean drought index (0-500),season rain vs normal (%),days at or above 35 °C
0,2000,154.8,7.4,5.6,1.7,0.2,D2,True,8.0,149.6,-21.1,3.0
1,2001,148.7,3.8,NaN,NaN,-4.8,D0,False,0.0,27.2,-8.3,3.0
2,2002,175.7,7.5,NaN,NaN,11.1,D0,False,0.0,27.3,-20.3,3.0
3,2003,165.3,5.2,3.0,2.1,3.4,D3,True,6.0,97.6,-14.3,10.0
4,2004,183.4,1.1,NaN,NaN,13.4,D0,False,0.0,0.7,5.6,0.0
5,2005,174.7,0.0,NaN,NaN,6.8,D1,False,0.0,13.1,-7.1,6.0
6,2006,171.9,3.2,-1.1,4.4,3.9,D2,True,2.0,80.7,-13.6,6.0
7,2007,177.8,3.9,NaN,NaN,6.3,D0,False,0.0,29.7,6.9,2.0
8,2008,154.0,-9.9,NaN,NaN,-9.0,none,False,0.0,0.0,43.7,1.0
9,2009,178.6,-0.2,NaN,NaN,4.4,none,False,0.0,0.0,-5.0,2.0


---
## 3. Evidence

Every table below is recomputed from the data each time the notebook runs, by the functions in `soil_planner/soil_tool.py`, with the same method and random seed as the analysis notebooks (03, 04 and 05). No number in a table is typed in. The sentences around the tables are written by hand and describe the pattern, not the exact figures.

### 3a. Does the drought-penalty score hold up on years it never saw? (notebook 05)

We built three scores from 2000–2015 only: the drought-penalty score, drought frequency, and soil water storage. We then asked whether they ranked counties correctly in the 2016–2025 drought years. "Spearman" is a rank correlation between the score and the actual drought penalty that year, signed so positive means the score got it right; chance is 0. "Top-third hit rate" is the share of the highest-priority third of counties that really were in the worst-losing third; chance is about 0.33. The range comes from resampling test years. The two placebo columns repeat the test where it should find nothing: in years with no drought anywhere in Iowa, and on the non-drought counties of every test year.

In [8]:
ia_split = st.compute_penalties(ia, split=True)        # held-out version: normals never cross 2015/2016
rk = st.penalty_ranking_results(ia_split)
for crop_label, crop in st.CROPS.items():
    info = rk["test_years"][crop]
    display(Markdown(f"**{crop_label}** — test years with at least {st.MIN_COUNTIES_PER_YEAR} drought counties: "
                     + ", ".join(f"{y} ({n})" for y, n in info["usable"].items())
                     + f". Years with no drought county (used for the first placebo): {', '.join(map(str, info['zero_drought']))}."))
    display(st.ranking_summary_table(rk["summary"], crop))

real = rk["summary"][rk["summary"]["test"] == st.REAL_TEST]
beats = real[real["range_above_chance"]][["crop", "score", "metric"]].values.tolist()
print("Computed verdict: score / metric combinations whose whole range sits above chance on held-out drought years:", beats or "none")

**Corn** — test years with at least 15 drought counties: 2017 (26), 2020 (48), 2021 (57), 2022 (41), 2023 (73), 2024 (18). Years with no drought county (used for the first placebo): 2016, 2019, 2025.

test                         chance   real: drought years placebo: zero-drought years placebo: non-D2+ counties, all test years n_years (real / placebo-0 / placebo-all)
score         metric                                                                                                                                                    
penalty_score spearman         0.00  +0.22 [+0.13, +0.31]        +0.17 [-0.25, +0.42]                      +0.04 [-0.17, +0.27]                                5 / 3 / 9
              top_third_hit    0.33  +0.45 [+0.36, +0.54]        +0.40 [+0.18, +0.52]                      +0.35 [+0.25, +0.46]                                5 / 3 / 9
drought_freq  spearman         0.00  -0.14 [-0.40, +0.15]        -0.05 [-0.19, +0.07]                      -0.03 [-0.17, +0.12]                                6 / 3 / 9
              top_third_hit    0.33  +0.31 [+0.14, +0.49]        +0.34 [+0.26, +0.41]                      +0.35 [+0.26, +0.43]                                6 / 3 / 9
storage       spearman         0.00  +0.17 [+0.05, +0.29]        +0.11 [+0.01, +0.30]                      -0.01 [-0.14, +0.14]                                6 / 3 / 9
              top_third_hit    0.33  +0.36 [+0.21, +0.46]        +0.29 [+0.17, +0.37]                      +0.30 [+0.25, +0.34]                                6 / 3 / 9

**Soybeans** — test years with at least 15 drought counties: 2017 (30), 2018 (17), 2020 (42), 2021 (50), 2022 (50), 2023 (52), 2024 (22). Years with no drought county (used for the first placebo): 2016, 2019, 2025.

test                         chance   real: drought years placebo: zero-drought years placebo: non-D2+ counties, all test years n_years (real / placebo-0 / placebo-all)
score         metric                                                                                                                                                    
penalty_score spearman         0.00  +0.16 [-0.01, +0.35]        -0.13 [-0.35, +0.09]                      +0.05 [-0.11, +0.22]                                7 / 2 / 7
              top_third_hit    0.33  +0.49 [+0.42, +0.54]        +0.20 [+0.20, +0.20]                      +0.32 [+0.24, +0.40]                                7 / 2 / 7
drought_freq  spearman         0.00  -0.19 [-0.53, +0.15]        -0.13 [-0.36, +0.10]                      +0.03 [-0.21, +0.27]                                7 / 2 / 7
              top_third_hit    0.33  +0.36 [+0.19, +0.53]        +0.25 [+0.19, +0.31]                      +0.36 [+0.24, +0.49]                                7 / 2 / 7
storage       spearman         0.00  +0.36 [+0.16, +0.56]        -0.01 [-0.04, +0.01]                      -0.02 [-0.16, +0.10]                                7 / 2 / 7
              top_third_hit    0.33  +0.53 [+0.44, +0.61]        +0.30 [+0.27, +0.33]                      +0.28 [+0.19, +0.35]                                7 / 2 / 7

Computed verdict: score / metric combinations whose whole range sits above chance on held-out drought years: [['CORN', 'penalty_score', 'spearman'], ['CORN', 'penalty_score', 'top_third_hit'], ['CORN', 'storage', 'spearman'], ['SOYBEANS', 'penalty_score', 'top_third_hit'], ['SOYBEANS', 'storage', 'spearman'], ['SOYBEANS', 'storage', 'top_third_hit']]


**Checks on the headline.** The three cells below compare the figures quoted in the headline with the values computed above. A failed check is printed as FAIL; the headline text is not changed automatically.

In [9]:
# Check 1: headline "about 45% (corn)" vs the computed corn top-third hit rate of the penalty score
corn_hit = real[(real["crop"] == "CORN") & (real["score"] == "penalty_score") & (real["metric"] == "top_third_hit")]["_mean"].iloc[0]
ok = abs(corn_hit - st.HEADLINE_CORN_HIT_RATE) <= 0.025
print(f"Check 1 {'PASS' if ok else 'FAIL'}: computed corn top-third hit rate = {corn_hit:.3f}; headline quotes {st.HEADLINE_CORN_HIT_RATE:.2f}")

Check 1 PASS: computed corn top-third hit rate = 0.453; headline quotes 0.45


In [10]:
# Check 2: headline "49% (soybeans)" vs the computed soybean top-third hit rate of the penalty score
soy_hit = real[(real["crop"] == "SOYBEANS") & (real["score"] == "penalty_score") & (real["metric"] == "top_third_hit")]["_mean"].iloc[0]
ok = abs(soy_hit - st.HEADLINE_SOY_HIT_RATE) <= 0.025
print(f"Check 2 {'PASS' if ok else 'FAIL'}: computed soybean top-third hit rate = {soy_hit:.3f}; headline quotes {st.HEADLINE_SOY_HIT_RATE:.2f}")

Check 2 PASS: computed soybean top-third hit rate = 0.485; headline quotes 0.49


In [11]:
# Check 3: headline "The corn ranking held up in every test year" = the corn rank correlation of the penalty score was positive in every usable test year
corn_py = rk["results"][("CORN", st.REAL_TEST)][0]["penalty_score"]["spearman"]
ok = bool((corn_py > 0).all())
print(f"Check 3 {'PASS' if ok else 'FAIL'}: corn per-year Spearman of the penalty score = "
      + ", ".join(f"{int(y)}: {v:+.2f}" for y, v in corn_py.items()) + f" ({int((corn_py > 0).sum())} of {len(corn_py)} years positive)")

Check 3 PASS: corn per-year Spearman of the penalty score = 2017: +0.34, 2020: +0.12, 2021: +0.09, 2022: +0.24, 2023: +0.31 (5 of 5 years positive)


**Reading it.** The drought-penalty score is the only score that beats chance on held-out drought years, and it does so for corn on both measures. For soybeans the top-third hit rate also beats chance while the rank correlation is weaker and its range touches zero. Drought frequency does not work for either crop. Storage alone ranks the 2016–2025 drought years correctly, but see 3b: that pattern was not consistent before 2016. The broad placebo (non-drought counties in every test year) is clean, so the persistent-county effect that broke the earlier test is gone; the zero-drought placebo has only two or three years and cannot be judged.

### 3b. Does soil water storage go with smaller drought penalties? (notebook 05)

For each year with at least 15 drought counties, the slope of the drought penalty on storage across those counties, in percentage points per extra 10 mm of storage, averaged over years with a range from resampling years. Positive would mean more storage, smaller penalty. The placebo column runs the same regression on non-drought county-years, where a real buffering effect should not appear.

In [12]:
step2, _ = st.penalty_storage_results(ia_split)
display(step2.drop(columns=["_mean", "_lo", "_hi"])
             .pivot(index=["crop", "spec"], columns="kind", values=["estimate [95% range]", "n_years", "county_years", "excludes_zero"])
             .swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)[["drought", "placebo"]])
n_excl = int(step2[(step2["kind"] == "drought") & step2["excludes_zero"]].shape[0])
print(f"Computed verdict: {n_excl} of {int((step2['kind'] == 'drought').sum())} drought-year storage estimates have a range that excludes zero; "
      f"placebo estimates excluding zero: {int(step2[(step2['kind'] == 'placebo') & step2['excludes_zero']].shape[0])}.")

kind                                       drought                                                  placebo                                   
                              estimate [95% range] n_years county_years excludes_zero  estimate [95% range] n_years county_years excludes_zero
crop     spec                                                                                                                                 
CORN     storage + longitude  +0.88 [-1.72, +3.75]      13          681         False  -0.24 [-1.09, +0.64]      23         1497         False
         storage only         +0.83 [-1.57, +3.44]      13          681         False  -0.30 [-1.13, +0.51]      23         1497         False
SOYBEANS storage + longitude  +1.38 [-0.92, +3.38]      14          720         False  +0.13 [-0.93, +1.23]      21         1362         False
         storage only         +1.81 [-0.08, +3.73]      14          720         False  -0.30 [-1.26, +0.60]      21         1362         False

Computed verdict: 0 of 4 drought-year storage estimates have a range that excludes zero; placebo estimates excluding zero: 0.


**Reading it.** Storage leans the right way in drought years and sits at zero in the placebo, which is the pattern a real buffering effect would produce, but every drought-year range includes zero. The per-year slopes are mixed before 2016 and mostly positive after. This is consistent with a small storage buffer, more so for soybeans, and it is not demonstrated. That is why the ranking is built on the drought penalty and not on storage.

### 3c. Earlier attempts, kept for the record (notebooks 03 and 04)

The first storage estimate used the dataset's own yield anomaly (a trend fitted on all 26 years, so in-sample) and found no clear effect. The first held-out ranking test built scores on a 2000–2015 trend and failed: no score beat chance, and the scores "predicted" losses even in years with no drought, because the trend extrapolated unevenly across the state and counties with a bad early record did better later regardless of weather. The drought-penalty score in 3a was designed to fix that and was then tested once.

In [13]:
storage03 = st.storage_effect_results(ia)
display(Markdown("**Notebook 03: in-sample storage estimate (pp of yield anomaly per +10 mm, drought years)**"))
display(storage03.drop(columns=["_mean", "_lo", "_hi"]).set_index(["crop", "specification"])[["estimate [95% range]", "drought years", "range excludes zero"]])
rank04 = st.ranking_test_results(ia)
display(Markdown("**Notebook 04: held-out test of trend-based scores (none beat chance) and its failed placebo**"))
display(rank04["summary"].set_index(["crop", "score"]))
display(rank04["placebo"].set_index(["crop", "year"]))

**Notebook 03: in-sample storage estimate (pp of yield anomaly per +10 mm, drought years)**

estimate [95% range]  drought years  range excludes zero
crop     specification                                                                         
CORN     A: storage only               +0.68 [-1.81, +3.43]             13                False
         B: + drought weeks, heat      +0.17 [-1.61, +1.71]             13                False
         C: + longitude (region)       -0.45 [-2.16, +1.17]             13                False
         D: + soil productivity        -0.32 [-1.90, +1.23]             13                False
         high minus low storage third  -0.24 [-4.64, +3.45]             11                False
SOYBEANS A: storage only               +1.11 [-0.25, +2.40]             14                False
         B: + drought weeks, heat      +0.73 [-0.55, +2.29]             14                False
         C: + longitude (region)       +0.38 [-1.18, +1.96]             14                False
         D: + soil productivity        +0.29 [-1.04, +1.62]             14                False
         high minus low storage third  +1.90 [-1.02, +4.72]             13                False

**Notebook 04: held-out test of trend-based scores (none beat chance) and its failed placebo**

Spearman [95% range] (chance 0) top-third hit rate [95% range] (chance 0.33)  test years  beats chance (whole range above chance)
crop     score                                                                                                                                         
CORN     drought_freq            -0.24 [-0.44, -0.09]                         +0.25 [+0.12, +0.36]           6                                    False
         drought_loss            +0.11 [-0.13, +0.36]                         +0.40 [+0.32, +0.48]           6                                    False
         storage                 -0.27 [-0.51, -0.04]                         +0.21 [+0.11, +0.29]           6                                    False
         combined                -0.07 [-0.25, +0.08]                         +0.29 [+0.22, +0.36]           6                                    False
SOYBEANS drought_freq            -0.13 [-0.37, +0.07]                         +0.29 [+0.16, +0.44]           7                                    False
         drought_loss            +0.02 [-0.11, +0.17]                         +0.38 [+0.30, +0.47]           7                                    False
         storage                 -0.10 [-0.21, +0.01]                         +0.31 [+0.23, +0.40]           7                                    False
         combined                -0.11 [-0.34, +0.15]                         +0.32 [+0.20, +0.45]           7                                    False

counties  Spearman drought_freq  Spearman drought_loss  Spearman storage  Spearman combined  storage slope per 10 mm
crop     year                                                                                                                      
CORN     2016        99                  -0.27                  -0.14             -0.50              -0.34                    -5.56
         2019        88                  -0.08                  -0.46             -0.24              -0.46                    -4.58
         2025        79                  -0.29                  -0.08             -0.55              -0.29                    -9.89
SOYBEANS 2016        99                  -0.19                  -0.00             -0.18              -0.08                    -5.18
         2019        82                  -0.42                  -0.05             -0.04              -0.42                    -3.15
         2025        73                  -0.39                  -0.02             -0.38              -0.38                    -6.29

**Checks on the methods note.** The two cells below verify the figures quoted in "How we tested this" against the values computed above.

In [14]:
# Check 4: methods note "fell to about a quarter of its raw size after all three" = soybean spec D estimate / spec A estimate
soy03 = storage03[storage03["crop"] == "SOYBEANS"].set_index("specification")["_mean"]
ratio = soy03["D: + soil productivity"] / soy03["A: storage only"]
ok = 0.2 <= ratio <= 0.3
print(f"Check 4 {'PASS' if ok else 'FAIL'}: soybean spec D / spec A = {soy03['D: + soil productivity']:+.3f} / {soy03['A: storage only']:+.3f} = {ratio:.2f} (about one quarter = 0.20 to 0.30)")

Check 4 PASS: soybean spec D / spec A = +0.289 / +1.106 = 0.26 (about one quarter = 0.20 to 0.30)


In [15]:
# Check 5: methods note "Of our 26 storage estimates, one range excluded zero; it rested on 5 years and pointed the opposite way
# from the matching regression" = notebook 03 main (10) + notebook 03 D3+ sensitivity (8) + notebook 05 Step 2 (8)
sens03 = st.storage_effect_sensitivity(ia)
n_total = len(storage03) + len(sens03) + len(step2)
excl = pd.concat([storage03.assign(source="nb03 main")[["source", "crop", "specification", "estimate [95% range]", "drought years", "range excludes zero"]],
                  sens03.assign(source="nb03 D3+ sensitivity: " + sens03["definition"])[["source", "crop", "specification", "estimate [95% range]", "drought years", "range excludes zero"]],
                  step2.assign(source="nb05 step 2: " + step2["kind"], specification=step2["spec"], **{"drought years": step2["n_years"], "range excludes zero": step2["excludes_zero"]})
                       [["source", "crop", "specification", "estimate [95% range]", "drought years", "range excludes zero"]]], ignore_index=True)
hits = excl[excl["range excludes zero"]]
ok_count = (n_total == 26) and (len(hits) == 1)
print(f"Check 5a {'PASS' if ok_count else 'FAIL'}: storage estimates = {n_total} (note says 26); ranges excluding zero = {len(hits)} (note says 1)")
if len(hits) == 1:
    h = hits.iloc[0]
    match = sens03[(sens03["crop"] == h["crop"]) & (sens03["definition"] == h["source"].split(": ", 1)[1]) & (sens03["specification"] != "high minus low storage third")].iloc[0]
    opposite = (h["estimate [95% range]"].startswith("-")) != (match["estimate [95% range]"].startswith("-"))
    ok_detail = (int(h["drought years"]) == 5) and opposite
    print(f"Check 5b {'PASS' if ok_detail else 'FAIL'}: the one estimate is {h['crop']}, {h['source']}, {h['specification']}: {h['estimate [95% range]']} on {int(h['drought years'])} years (note says 5); "
          f"matching regression on the same subset: {match['estimate [95% range]']} -> opposite sign: {opposite}")
display(excl.set_index(["source", "crop", "specification"]))

Check 5a PASS: storage estimates = 26 (note says 26); ranges excluding zero = 1 (note says 1)
Check 5b PASS: the one estimate is CORN, nb03 D3+ sensitivity: D3+ excl. 2012, high minus low storage third: -1.51 [-2.85, -0.35] on 5 years (note says 5); matching regression on the same subset: +0.85 [-0.70, +2.73] -> opposite sign: True


estimate [95% range]  drought years  range excludes zero
source                               crop     specification                                                                         
nb03 main                            CORN     A: storage only               +0.68 [-1.81, +3.43]             13                False
                                              B: + drought weeks, heat      +0.17 [-1.61, +1.71]             13                False
                                              C: + longitude (region)       -0.45 [-2.16, +1.17]             13                False
                                              D: + soil productivity        -0.32 [-1.90, +1.23]             13                False
                                              high minus low storage third  -0.24 [-4.64, +3.45]             11                False
                                     SOYBEANS A: storage only               +1.11 [-0.25, +2.40]             14                False
                                              B: + drought weeks, heat      +0.73 [-0.55, +2.29]             14                False
                                              C: + longitude (region)       +0.38 [-1.18, +1.96]             14                False
                                              D: + soil productivity        +0.29 [-1.04, +1.62]             14                False
                                              high minus low storage third  +1.90 [-1.02, +4.72]             13                False
nb03 D3+ sensitivity: D3+            CORN     C: + longitude (region)       +1.41 [-0.29, +3.17]              7                False
                                              high minus low storage third  +0.53 [-2.40, +4.83]              6                False
nb03 D3+ sensitivity: D3+ excl. 2012 CORN     C: + longitude (region)       +0.85 [-0.70, +2.73]              6                False
                                              high minus low storage third  -1.51 [-2.85, -0.35]              5                 True
nb03 D3+ sensitivity: D3+            SOYBEANS C: + longitude (region)       +0.84 [-0.89, +2.22]              8                False
                                              high minus low storage third  -0.68 [-4.88, +2.39]              7                False
nb03 D3+ sensitivity: D3+ excl. 2012 SOYBEANS C: + longitude (region)       +0.58 [-1.35, +2.08]              7                False
                                              high minus low storage third  -0.68 [-5.67, +2.90]              6                False
nb05 step 2: drought                 CORN     storage only                  +0.83 [-1.57, +3.44]             13                False
                                              storage + longitude           +0.88 [-1.72, +3.75]             13                False
nb05 step 2: placebo                 CORN     storage only                  -0.30 [-1.13, +0.51]             23                False
                                              storage + longitude           -0.24 [-1.09, +0.64]             23                False
nb05 step 2: drought                 SOYBEANS storage only                  +1.81 [-0.08, +3.73]             14                False
                                              storage + longitude           +1.38 [-0.92, +3.38]             14                False
nb05 step 2: placebo                 SOYBEANS storage only                  -0.30 [-1.26, +0.60]             21                False
                                              storage + longitude           +0.13 [-0.93, +1.23]             21                False

In [16]:
# "How we tested this" lives only in soil_planner/soil_tool.py (METHODS_NOTE).
display(Markdown(st.METHODS_NOTE))

### How we tested this

**Comparing counties fairly.** In Iowa, soil water storage runs high in the west and low in the east, so comparing high- and low-storage counties also compares regions, weather and farming. We added controls one at a time; each one reduced the soybean association, which fell to about a quarter of its raw size after all three. *(Bueno de Mesquita & Fowler, ch. 10; Pearl & Mackenzie, ch. 4)*

**Comparing each county with itself.** The drought penalty compares a county's yield in a drought year with its own nearby normal years, relative to the rest of the state. This cancels anything permanent about a county. Our first test failed because extrapolated yield trends differed by region, which breaks the assumption this kind of comparison relies on. *(Bueno de Mesquita & Fowler, ch. 13; Gelman, Hill & Vehtari, §21.4)*

**Checking the test itself.** We ran each test in years with no drought, where a real drought score should predict nothing. Our first test "predicted" losses there, so we rejected it. The drought-penalty score showed nothing in the non-drought counties of every test year. The stricter check, using only years with no drought anywhere in Iowa, covers just two or three years and is too short to judge.

**Reversion to the mean.** Counties with the worst early drought record did relatively better later even without any real change, which is why raw drought frequency ranked counties in the wrong direction. *(Bueno de Mesquita & Fowler, ch. 8; Gelman, Hill & Vehtari, §6.5)*

**Uncertainty.** Our ranges come from resampling whole years, because counties in the same year share the same weather. A range that includes zero means the data can't rule out "no effect," not that there is none. Of our 26 storage estimates, one range excluded zero; it rested on 5 years and pointed the opposite way from the matching regression, so we don't treat it as a finding. *(Bueno de Mesquita & Fowler, ch. 6–7; Gelman, Hill & Vehtari, §4.5, §5.4)*

**Why some counties are "lower confidence."** Every score mixes real signal with noise, and a score built on only a few drought years carries more noise. Extreme scores built on little data are the most likely to drift back toward average in future droughts, which is why we flag counties whose score rests on fewer than 6 drought years. *(Bueno de Mesquita & Fowler, ch. 8)*

**References.** Bueno de Mesquita, E. & Fowler, A. (2021). *Thinking Clearly with Data.* Princeton University Press, ch. 6–8, 10, 13, 15–17. · Gelman, A., Hill, J. & Vehtari, A. (2020). *Regression and Other Stories.* Cambridge University Press. · Pearl, J. & Mackenzie, D. (2018). *The Book of Why.* Basic Books.

---
## 4. Limits of this evidence


In [17]:
# The limits text lives only in soil_planner/soil_tool.py (LIMITS_NOTE); the Streamlit app shows the same text.
display(Markdown(st.LIMITS_NOTE))

- **Association, not program effect.** The drought penalty and soil water storage are properties of counties as they are. Comparing counties is not the same as measuring what a conservation practice would do on a given field. Nothing here estimates a program's effect, and the priority list shows where drought has hurt most, not where a program would help most.
- **Few test years, and no statewide drought among them.** The held-out test rests on 5 to 7 drought years (2017–2024), all partial-state events. No 2012-type drought that covered every county is in the test set, and that is the case the office cares most about. The ranges in section 3 are wide for that reason.
- **The score was redesigned once and tested once.** The first ranking test (notebook 04) failed its placebo check. The drought-penalty score was designed to fix that and then tested a single time on the same 2016–2025 years. One pass is encouraging, not proof; a result chosen after one failure deserves more caution than a result that was right the first time.
- **The zero-drought placebo is too thin to judge.** Only three test years (two for soybeans) had no drought anywhere in Iowa. The broader placebo on non-drought counties in every test year is clean, but the strict one cannot confirm or deny anything.
- **Storage could not be separated from zero.** Storage leans toward smaller drought penalties and shows nothing in the placebo, but every range includes zero, Iowa's storage range is narrow (about 154 to 206 mm), and high-storage counties sit in the west while low-storage counties sit in the east, so storage and region are entangled.
- **Irrigation.** The data say Iowa is rainfed only through documentation. There is no irrigation column to confirm it county by county.
- **Yield data thin out after 2022.** Fewer counties report yields in recent years, so recent drought penalties are missing for more counties, especially soybeans.
- **Extra soil and practice data are context only.** The surface-soil properties (September 2026 SSURGO release) and the Census of Agriculture practice shares (2017 and 2022) shown in the county panel were not tested against drought losses and play no part in the ranking or the evidence.
- Our score measures where drought losses have occurred, not where soil programs would reduce them. That is only part of the office's mission (Bueno de Mesquita & Fowler, ch. 16).
- The lessons come from Iowa, 2000–2025. Future droughts, new practices, or other states may not behave the same way.
- A pre-registered replication on Nebraska's rain-fed counties could not be run: irrigation leaves only 13–18 mostly-dryland counties, below our pre-set minimum of 20.

---
## 5. Next steps


In [18]:
# The next-steps text lives only in soil_planner/soil_tool.py (NEXT_STEPS_NOTE); the Streamlit app shows the same text.
display(Markdown(st.NEXT_STEPS_NOTE))

**Next steps**
- **Test whether programs work.** Once the office funds programs in some counties, compare how drought penalties change in funded versus unfunded counties, before and after. That is a true difference-in-differences test of program impact (Bueno de Mesquita & Fowler, ch. 13).
- **Use the lever, not just the context.** Compare the change in cover-crop and no-till adoption between the 2017 and 2022 Censuses with the change in each county's drought penalty.
- **Measure drought timing.** The master dataset already includes a seasonal mean drought intensity (`mean_dsci`, 0–500 scale) that could replace the single D2+ threshold; the weekly Drought Monitor maps would add timing within the season, to focus on drought during critical growth months.
- **Limited evidence is not a reason to do nothing.** The storage result was inconclusive, but not acting is also a decision; local evaluation of practices in priority counties is a reasonable response (Bueno de Mesquita & Fowler, ch. 17).

---
## Data credits


In [19]:
# The credits text lives only in soil_planner/soil_tool.py (DATA_CREDITS); the app's About note shows the same text.
display(Markdown(st.DATA_CREDITS))

**Data credits.** Credit the sources when presenting results from this data (wording from the hackathon README):
- **USDA NASS Quick Stats** (county yields; Census of Agriculture conservation practices): This product uses the NASS API but is not endorsed or certified by NASS.
- **NOAA ACIS** (weather): Weather data from the Applied Climate Information System (ACIS), NOAA Regional Climate Centers.
- **U.S. Drought Monitor** (drought): The U.S. Drought Monitor is jointly produced by the National Drought Mitigation Center at the University of Nebraska-Lincoln, the United States Department of Agriculture, and the National Oceanic and Atmospheric Administration.
- **USDA NRCS SSURGO / Soil Data Access** (soil): Soil Survey Staff, Natural Resources Conservation Service, United States Department of Agriculture. Soil Survey Geographic (SSURGO) Database, via Soil Data Access.
- **Data to Harvest hackathon master dataset** (Delaware State University, October 2026): the county x crop x year table that combines the sources above.